# Adaptive Guardrail Layer (AGL) - Data Cleaning Notebook

In [ ]:
import os
import re
import unicodedata
from pathlib import Path
import pandas as pd
import numpy as np
pd.set_option("display.max_colwidth", None)

INPUT_PATH = OUTPUT_PATH = "../../data/processed/"

input_file = Path(INPUT_PATH) / "dataset_merged.csv"
if not os.path.exists(input_file):
    raise FileNotFoundError(f"Could not find input file: {input_file}")

print(f"Input file found: {input_file}")

### Load the dataset

In [ ]:
# load the dataset
df = pd.read_csv(input_file)

# print basic information
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

# display the first few rows
df.head()

### Inspect the schema and data types

In [ ]:
# print dataframe information
print(df.info())

# show data types only
print("\nData types:")
print(df.dtypes)

### Assess missing values

In [ ]:
# count missing values in each column
missing_counts = df.isna().sum()

# calculate percentage of missing values
missing_percent = (df.isna().mean() * 100).round(2)

missing_summary = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_percent": missing_percent
})

missing_summary

### Inspect target labels and source dataset values

In [ ]:
# show unique label values
print("Label distribution:")
print(df["label"].value_counts(dropna=False))

print("\nSource dataset distribution:")
print(df["source_dataset"].value_counts(dropna=False))

### Inspect duplicate records

In [ ]:
# count exact duplicate rows
full_duplicates = df.duplicated().sum()

# count duplicate prompts only
prompt_duplicates = df.duplicated(subset=["prompt"]).sum()

print("Exact duplicate rows:", full_duplicates)
print("Duplicate prompts:", prompt_duplicates)

### Create a copy for cleaning

In [ ]:
# create a working copy of the original dataframe
df_clean = df.copy()

print("Working copy created.")
print("Shape:", df_clean.shape)

### Remove rows with missing or empty prompts

In [ ]:
# record starting row count
rows_before = len(df_clean)

# drop rows where prompt is missing
df_clean = df_clean.dropna(subset=["prompt"]).copy()

# convert prompt to string and strip leading/trailing whitespace
df_clean["prompt"] = df_clean["prompt"].astype(str).str.strip()

# remove rows with empty prompt strings
df_clean = df_clean[df_clean["prompt"] != ""].copy()

rows_after = len(df_clean)

print("Rows before removing missing/empty prompts:", rows_before)
print("Rows after removing missing/empty prompts:", rows_after)
print("Rows removed:", rows_before - rows_after)

### Normalize prompt text

In [ ]:
def normalize_text(text: str) -> str:
    if pd.isna(text):
        return text
    
    # normalize unicode characters to a consistent representation
    text = unicodedata.normalize("NFKC", text)
    
    # standardize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    
    # collapse repeated whitespace but preserve single spaces
    text = re.sub(r"\s+", " ", text)
    
    # remove leading/trailing whitespace
    text = text.strip()
    
    return text

# apply normalization to the prompt column
df_clean["prompt"] = df_clean["prompt"].apply(normalize_text)

# quick sample view
df_clean[["prompt", "label", "source_dataset"]].head()

### Remove exact duplicate rows

In [ ]:
# count rows before removing exact duplicates
rows_before = len(df_clean)

# remove exact duplicate rows
df_clean = df_clean.drop_duplicates().copy()

rows_after = len(df_clean)

print("Rows before removing exact duplicates:", rows_before)
print("Rows after removing exact duplicates:", rows_after)
print("Exact duplicates removed:", rows_before - rows_after)

### Remove duplicate prompts

In [ ]:
# check whether any prompt appears with more than one label
label_conflicts = (
    df_clean.groupby("prompt")["label"]
    .nunique()
    .reset_index(name="n_unique_labels")
)

conflicting_prompts = label_conflicts[label_conflicts["n_unique_labels"] > 1]

print("Prompts with conflicting labels:", len(conflicting_prompts))

# display a few examples if conflicts exist
if len(conflicting_prompts) > 0:
    sample_conflicts = conflicting_prompts["prompt"].head(5).tolist()
    display(df_clean[df_clean["prompt"].isin(sample_conflicts)].sort_values("prompt"))

# drop conflicting prompts
conflicting_prompt_list = conflicting_prompts["prompt"].tolist()
df_clean = df_clean[~df_clean["prompt"].isin(conflicting_prompt_list)]
print("Dataset size after removing conflicts:", df_clean.shape)

### Cleaning summary table

In [ ]:
# Build a compact cleaning summary
cleaning_summary = pd.DataFrame({
    "metric": [
        "original_rows",
        "cleaned_rows",
        "columns",
        "missing_prompts_remaining",
        "missing_labels_remaining",
        "exact_duplicate_rows_remaining",
        "duplicate_prompts_remaining",
        "conflicting_prompt_count",
        "label_0_count",
        "label_1_count"
    ],
    "value": [
        len(df),
        len(df_clean),
        df_clean.shape[1],
        df_clean["prompt"].isna().sum(),
        df_clean["label"].isna().sum(),
        df_clean.duplicated().sum(),
        df_clean.duplicated(subset=["prompt"]).sum(),
        len(conflicting_prompts),
        int((df_clean["label"] == 0).sum()),
        int((df_clean["label"] == 1).sum())
    ]
})

cleaning_summary

### Final validation preview

In [ ]:
# display a random sample of cleaned rows
df_clean.sample(10, random_state=42)[["prompt", "label", "source_dataset"]]

### Save cleaned dataset

In [ ]:
# save the cleaned dataset to CSV
output_file = Path(OUTPUT_PATH) / "dataset_cleaned.csv"
df_clean.to_csv(output_file, index=False)

# verify file creation
if output_file.exists():
    print(f"File created successfully: {output_file}")
else:
    raise FileNotFoundError(f"File was NOT created: {output_file}")